In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

# Base directory (where notebook is executed)
BASE_DIR = Path(os.getcwd())
print(f"Base directory: {BASE_DIR}")
run = "2"
RASDAMAN_DIR = BASE_DIR / f"results/rasdaman/run{str(run)}"
ODC_DIR = BASE_DIR / f"results/odc/run{str(run)}"
OUTPUT_DIR = BASE_DIR / f"results/run{str(run)}"

OUTPUT_DIR.mkdir(exist_ok=True)

FILE_PREFIX = "results_execution"

COLORS = {
    "Rasdaman": "#1f77b4",
    "OpenDataCube": "#ff7f0e",
}

Base directory: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis


In [13]:
def extract_run_number(filename: str, prefix: str):
    pattern = rf"{prefix}(\d+)_\d+\.csv"
    match = re.match(pattern, filename)
    return int(match.group(1)) if match else None


def group_files_by_run(directory: Path, prefix: str):
    files = list(directory.glob(f"{prefix}*.csv"))
    runs = {}
    
    for file in files:
        run_num = extract_run_number(file.name, prefix)
        if run_num is not None:
            runs.setdefault(run_num, []).append(file)
    
    return runs


def find_matching_runs(dir1: Path, dir2: Path, prefix: str):
    ras_runs = group_files_by_run(dir1, prefix)
    odc_runs = group_files_by_run(dir2, prefix)
    
    common_runs = set(ras_runs.keys()) & set(odc_runs.keys())
    
    return [
        (run_num, ras_runs[run_num], odc_runs[run_num])
        for run_num in sorted(common_runs)
    ]

In [6]:
def load_results(path: Path, sut_name: str):
    df = pd.read_csv(path)
    df["SUT"] = sut_name
    df["file"] = path.name
    return df


def load_run_results(file_list: list, sut_name: str):
    return pd.concat(
        [load_results(file_path, sut_name) for file_path in file_list],
        ignore_index=True
    )

In [14]:
def create_summary_table(df_combined: pd.DataFrame, run_number: int):
    summary_rows = []

    for sut in df_combined["SUT"].unique():
        sut_df = df_combined[df_combined["SUT"] == sut]

        summary_rows.append({
            "SUT": sut,
            "Execution time (ms)": round(sut_df["latency_ms"].sum(), 2),
            "ω.90 (ms)": round(sut_df["latency_ms"].quantile(0.90), 2),
            "ω.99 (ms)": round(sut_df["latency_ms"].quantile(0.99), 2),
            "Avg. latency (ms)": round(sut_df["latency_ms"].mean(), 2),
        })

    df_summary = pd.DataFrame(summary_rows)
    output_path = OUTPUT_DIR / f"run{run_number}_summary.csv"
    df_summary.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    return df_summary


def plot_boxplot(df_combined: pd.DataFrame, run_number: int):
    plt.figure(figsize=(6, 4))

    data = [
        df_combined[df_combined["SUT"] == sut]["latency_ms"]
        for sut in COLORS.keys()
    ]

    bp = plt.boxplot(
        data,
        patch_artist=True,
        labels=COLORS.keys(),
        widths=0.5,
        showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "black", "markeredgecolor": "black"},
    )

    for patch, sut in zip(bp["boxes"], COLORS.keys()):
        patch.set_facecolor(COLORS[sut])
        patch.set_alpha(0.6)
        patch.set_edgecolor("black")
        patch.set_linewidth(1.5)

    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.ylabel("Latency (ms)")
    plt.tight_layout()

    output_path = OUTPUT_DIR / f"run{run_number}_latency_boxplot.png"
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"Saved: {output_path}")

In [15]:
def create_overall_summary(df_all: pd.DataFrame):
    summary_rows = []

    for sut in df_all["SUT"].unique():
        sut_df = df_all[df_all["SUT"] == sut]

        summary_rows.append({
            "SUT": sut,
            "Execution time (ms)": round(sut_df["latency_ms"].sum(), 2),
            "ω.90 (ms)": round(sut_df["latency_ms"].quantile(0.90), 2),
            "ω.99 (ms)": round(sut_df["latency_ms"].quantile(0.99), 2),
            "Avg. latency (ms)": round(sut_df["latency_ms"].mean(), 2),
        })

    df_summary = pd.DataFrame(summary_rows)
    output_path = OUTPUT_DIR / "overall_summary.csv"
    df_summary.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")


def plot_overall_boxplot(df_all: pd.DataFrame):
    plt.figure(figsize=(6, 4))

    data = [
        df_all[df_all["SUT"] == sut]["latency_ms"]
        for sut in COLORS.keys()
    ]

    bp = plt.boxplot(
        data,
        patch_artist=True,
        labels=COLORS.keys(),
        widths=0.5,
        showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "black", "markeredgecolor": "black"},
    )

    for patch, sut in zip(bp["boxes"], COLORS.keys()):
        patch.set_facecolor(COLORS[sut])
        patch.set_alpha(0.6)

    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.ylabel("Latency (ms)")
    plt.tight_layout()

    output_path = OUTPUT_DIR / "overall_latency_boxplot.png"
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"Saved: {output_path}")

In [17]:
matching_runs = find_matching_runs(RASDAMAN_DIR, ODC_DIR, FILE_PREFIX)
all_combined = []

if not matching_runs:
    print("No matching runs found!")
else:
    for run_num, ras_files, odc_files in matching_runs:
        df_ras = load_run_results(ras_files, "Rasdaman")
        df_odc = load_run_results(odc_files, "OpenDataCube")

        df_combined = pd.concat([df_ras, df_odc], ignore_index=True)
        df_combined = df_combined[df_combined["status_code"] == 200]

        all_combined.append(df_combined)

        plot_boxplot(df_combined, run_num)
        create_summary_table(df_combined, run_num)

        print(f"Run {run_num}: {len(ras_files)} Rasdaman, {len(odc_files)} ODC files")

    df_all = pd.concat(all_combined, ignore_index=True)

    plot_overall_boxplot(df_all)
    create_overall_summary(df_all)

    print(f"\nAll outputs saved in: {OUTPUT_DIR.resolve()}")

Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run1_latency_boxplot.png
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run1_summary.csv
Run 1: 20 Rasdaman, 19 ODC files
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run2_latency_boxplot.png
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run2_summary.csv
Run 2: 20 Rasdaman, 20 ODC files
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run3_latency_boxplot.png
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run3_summary.csv
Run 3: 20 Rasdaman, 20 ODC files
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run4_latency_boxplot.png
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run4_summary.csv
Run 4: 20 Rasdaman, 20 ODC files
Saved: /Users/gov/Prog/git/bachelor-thesis-benchsetup/analysis/results/run2/run5_latency_box